# Byg din egen sprogmodel: en lille transformer (nano-GPT)

Velkommen! I denne notebook leger I med en **transformer** -- den slags model der ligger bag fx
ChatGPT. Vores udgave er lille nok til at køre i Google Colab, og den skriver fantasy-historier
i formatet:

```
Navn: det personen siger
Navn: *det personen gør*
```

**Jeres opgave:** I får en færdig **basis-model** og **finetuner** den, så den skifter stil.
Undervejs kan I skrue på en masse knapper og se hvad der sker.

> **VIGTIGT -- vælg GPU!** Øverst i Colab: **Runtime -> Change runtime type -> T4 GPU**.
> Ellers bliver det meget langsomt.

## Hvordan virker en transformer?

En **sprogmodel** har egentlig kun ét job: at gætte det **næste tegn**. Giver man den
`Sir Aldric: Vi må n`, gætter den at det næste tegn nok er `å`. Den hænger gættet på og gætter
igen -- og sådan skriver den hele historier, ét tegn ad gangen.

For at gætte godt går teksten gennem fire trin:

**1. Fra tegn til tal (tokens + embeddings).**
Computeren kan ikke regne på bogstaver. Så hvert tegn får først et nummer (en *token*) og
derefter en lille liste af tal -- en *vektor* -- som modellen selv lærer. Tegn der bruges ens,
ender med vektorer der ligner hinanden.

**2. Hvor i teksten er vi? (positioner).**
Rækkefølge betyder alt: `konge` og `gnoke` har de samme bogstaver. Derfor får hver position
også sine egne tal, så modellen ved hvad der kom først.

**3. Kig tilbage på det vigtige (attention).**
Dette er hjertet i en transformer. For hvert tegn kigger modellen **tilbage** på de tidligere
tegn og afgør hvilke der er vigtige lige nu. Skal den skrive videre efter `Sir Aldric: `, hjælper
det at "huske" navnet og kolonet, så den ved at nu kommer en replik. Modellen kigger **aldrig
fremad** -- den kender jo ikke fremtiden, præcis som når du selv skriver.

> **Billede:** Forestil dig at du læser en sætning og hele tiden kan kigge tilbage på de ord der
> betød mest. Det er præcis hvad attention gør -- bare med tal.

**4. Gæt næste tegn -- og gentag.**
Til sidst laver modellen en sandsynlighed for hvert muligt næste tegn og vælger ét. Trin 2-3
gentages i flere **lag** (blocks), så modellen kan fange mere og mere komplekse mønstre.

**Hvordan lærer den?** Under træning gætter modellen næste tegn rigtig mange gange. Hver gang
måler vi hvor forkert den var (et tal kaldet **loss**) og justerer dens tal en lillebitte smule.
Lavere loss = bedre model.

**Hvad er finetuning?** Det er at tage en model der allerede kan "sproget" og give den lidt ekstra
træning på *nyt* data, så den skifter stil -- uden at starte forfra. Det er det, I skal lege med.

> Du behøver **ikke** forstå matematikken bag. Husk bare idéen:
> **tegn -> tal -> husk rækkefølgen -> kig tilbage på det vigtige -> gæt næste tegn -> gentag.**

In [1]:
# Colab har normalt alt installeret. Hvis torch mangler, installeres det automatisk.
try:
    import torch
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"])
    import torch

import os, math, time
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
from tqdm.auto import tqdm

/home/mimodekj/.conda/envs/ML/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Kontrolpanel

Her samler vi **alle** knapperne. I behøver ikke forstå dem alle endnu. To vigtige er flagene
`RETRAIN_BASE` og `DO_FINETUNE` -- de styrer hvad der skal køres, så I **slipper for at slette
filer** for at prøve igen.

In [2]:
# ============================ KONTROLPANEL ============================
# Aendr et tal eller et flag, koer cellerne igen, og se hvad der sker!

# --- Hvad skal koeres? (saa du slipper for at slette modeller manuelt) ---
RETRAIN_BASE = False   # False = brug den faerdige base_model.pth.  True = traen basis HELT forfra.
DO_FINETUNE  = True    # True = finetun paa dit valgte data.        False = spring finetuning over.

# --- Modellens stoerrelse (skal vaere ENS for basis OG finetuning!) ---
BLOCK_SIZE = 256     # hvor mange tegn modellen kan se ad gangen (kontekstvindue)
BATCH_SIZE = 64      # hvor mange tekststykker den traener paa samtidig
N_EMBD     = 384     # stoerrelsen paa hver tegn-vektor
N_HEAD     = 6       # antal opmaerksomheds-hoveder (heads)
N_LAYER    = 6       # antal transformer-lag (blocks)
DROPOUT    = 0.2     # lidt tilfaeldig "stoej" der modvirker udenadslaere

# --- Basis-traening (instruktoeren har typisk lavet base_model.pth paa forhaand) ---
LEARNING_RATE = 3e-4  # hvor store skridt modellen tager naar den laerer
MAX_ITERS     = 5000  # antal traenings-skridt for basis-modellen

# --- Finetuning (DIN opgave -- holdes under ~10 min paa en T4 GPU) ---
FINETUNE_LR    = 1e-4  # lavere end basis: vi vil kun "justere" modellen
FINETUNE_ITERS = 1000  # faerre skridt -- vi starter jo fra en god model

# --- Generering (naar modellen skriver historier) ---
TEMPERATURE = 0.8   # lav = forsigtig/forudsigelig, hoej = vild/kreativ
TOP_K       = 20    # modellen vaelger kun blandt de TOP_K mest sandsynlige tegn
GEN_TOKENS  = 600   # hvor mange tegn der genereres

# --- Teknisk ---
EVAL_INTERVAL = 250          # hvor tit vi maaler hvor godt det gaar
EVAL_ITERS    = 20           # antal stikproever til maalingen
EARLY_STOPPING_PATIENCE = 5  # stop hvis val-loss ikke bliver bedre saa mange gange i traek
BETA1, BETA2 = 0.9, 0.95     # indstillinger til AdamW-optimizeren

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1337)

print(f"Bruger: {DEVICE}")
if DEVICE == 'cpu':
    print("OBS: Du koerer paa CPU. Vaelg GPU: Runtime -> Change runtime type -> T4 GPU.")

Bruger: cuda


## 1. Data: tre fantasy-datasæt

Vi træner på tekst i formatet `Navn: replik` / `Navn: *handling*`. Der findes tre færdige filer
(lavet af underviseren -- I skal bare bruge dem):

| Fil | Stil |
|-----|------|
| `data_base.txt` | Relativt seriøst fantasy -> den gode **basis-model** |
| `data_finetune.txt` | Samme verden + gaming/internet-jokes (**standard** til finetuning) |
| `data_cooked.txt` | Totalt "kogt" brainrot (til når I vil have det helt vildt) |

**Fast vokabular:** Vi bestemmer på forhånd præcis hvilke tegn modellen kender (`VOCAB_CHARS`).
Det gør at en færdigtrænet model altid passer -- også når I tilføjer jeres eget data.

In [3]:
# Det FASTE vokabular -- skal matche det i generate_datasets.py.
VOCAB_CHARS = (
    "\n "                            # linjeskift og mellemrum
    "abcdefghijklmnopqrstuvwxyz"     # smaa bogstaver
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"     # store bogstaver
    "æøåÆØÅ"                          # danske bogstaver
    "0123456789"                     # tal
    ".,!?:;-'\"*()[]"                # tegnsaetning, * (handling) og [] (kontekst)
)
VOCAB_SET = set(VOCAB_CHARS)

def filter_to_vocab(text):
    """Fjerner tegn der ikke er i vokabularet. Returnerer (renset_tekst, antal_fjernede)."""
    kept = [c for c in text if c in VOCAB_SET]
    return "".join(kept), len(text) - len(kept)

# Byg tokenizer ud fra det FASTE vokabular (ikke ud fra data!).
chars = sorted(VOCAB_SET)
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # tegn -> tal
itos = {i: ch for i, ch in enumerate(chars)}   # tal  -> tegn

def encode(s):
    """Tekst -> liste af tal."""
    return [stoi[c] for c in s]

def decode(l):
    """Liste af tal -> tekst."""
    return ''.join(itos[i] for i in l)

# Tjek at datafilerne ligger her (ellers: upload dem, eller koer generate_datasets.ipynb).
for fn in ['data_base.txt', 'data_finetune.txt', 'data_cooked.txt']:
    if not os.path.exists(fn):
        raise FileNotFoundError(
            f"Filen '{fn}' mangler. Upload de tre data-filer til Colab "
            f"(eller bed din underviser om dem).")

def load_dataset(path, extra_text=""):
    """Laeser en datasaet-fil, tilfoejer evt. ekstra tekst, og deler i train/val (90/10)."""
    text = open(path, encoding='utf-8').read()
    if extra_text:
        clean, removed = filter_to_vocab(extra_text)
        if removed:
            print(f"(Fjernede {removed} tegn fra dit ekstra-data som ikke er i vokabularet.)")
        text = text + "\n" + clean
    data = torch.tensor(encode(text), dtype=torch.long)
    n = int(0.9 * len(data))
    return data[:n], data[n:]

print(f"Vokabular: {vocab_size} tegn. Alle tre datafiler er fundet.\n")
print("Et lille kig i basis-data:\n")
print(open('data_base.txt', encoding='utf-8').read()[:300])

Vokabular: 84 tegn. Alle tre datafiler er fundet.

Et lille kig i basis-data:

Vandreren Rurik: Healeren Tove, du tager teten mod porten.
Stifinderen Aksel: Tag dit våben frem, Barden Frode.
Druiden Siv: Kongeriget regner med os nu.
Stifinderen Aksel: Vi klarede det sidste, vi klarer også smedjen.
Stifinderen Aksel: Jeg går først - dæk min ryg.
Smeden Bjørn: Tag det gamle bann


## 2. Tokenizer: fra tekst til tal

Computeren regner kun på tal, så vi giver hvert tegn et nummer. `encode` laver tekst -> tal, og
`decode` laver tal -> tekst. Det er den simpleste slags tokenizer: **ét tegn = ét tal** (*char-level*).

In [4]:
sample = "Sir Aldric: Vi må nå frem til borgen!"
print("Tekst   :", sample)
print("Tal     :", encode(sample))
print("Tilbage :", decode(encode(sample)))
assert decode(encode(sample)) == sample, "encode/decode passer ikke!"
print("Tokenizer virker!")

Tekst   : Sir Aldric: Vi må nå frem til borgen!
Tal     : [42, 60, 69, 1, 24, 63, 55, 69, 60, 54, 21, 1, 45, 60, 1, 64, 81, 1, 65, 81, 1, 57, 69, 56, 64, 1, 71, 60, 63, 1, 53, 66, 69, 58, 56, 65, 2]
Tilbage : Sir Aldric: Vi må nå frem til borgen!
Tokenizer virker!


## 3. Vi bygger modellen

Nedenfor bygger vi præcis de dele vi beskrev øverst: **embeddings** (tegn -> tal),
**attention** (kig tilbage) og **lagene** (blocks). Læs gerne kommentarerne -- eller kør bare
cellerne og leg med modellen bagefter.

In [5]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    block_size: int      # kontekstvindue (antal tegn)
    vocab_size: int      # antal forskellige tegn
    n_layer: int         # antal transformer-lag
    n_head: int          # antal opmaerksomheds-hoveder
    n_embd: int          # stoerrelsen paa hver tegn-vektor
    dropout: float = 0.2
    bias: bool = True

class LayerNorm(nn.Module):
    """Holder tallene i et paent leje, saa traeningen er stabil."""
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    """Selv-opmaerksomhed: hvert tegn kigger BAGUD paa de tidligere tegn."""
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # Beregner query, key og value for alle hoveder paa én gang.
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # "Maske" der forhindrer at man kigger FREM i teksten (kun bagud er tilladt).
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                             .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size()  # batch, laengde, vektor-stoerrelse
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # Hvor godt passer hvert tegn til hvert tidligere tegn?
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))  # skjul fremtiden
        att = F.softmax(att, dim=-1)        # lav det om til vaegte der summer til 1
        att = self.attn_dropout(att)
        y = att @ v                          # vaegtet blanding af de tidligere tegn
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    """Et lille fremad-netvaerk der "taenker" lidt videre paa hvert tegn."""
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    """Et transformer-lag: foerst opmaerksomhed (kig tilbage), saa lidt "taenkning" (MLP)."""
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # opmaerksomhed (+ "genvej"/residual)
        x = x + self.mlp(self.ln_2(x))    # MLP (+ "genvej"/residual)
        return x

In [6]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),   # tegn -> vektor
            wpe  = nn.Embedding(config.block_size, config.n_embd),   # position -> vektor
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # deler vaegte (sparer parametre)

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        n_params = sum(p.numel() for p in self.parameters())
        print(f"Modellen har {n_params/1e6:.2f} millioner parametre.")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Sekvensen er for lang ({t} > {self.config.block_size})"
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)   # tegn-vektorer
        pos_emb = self.transformer.wpe(pos)   # positions-vektorer
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])   # kun sidste position naar vi genererer
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, context_len=0):
        """Skriver videre, ét tegn ad gangen.

        context_len: antal tegn forrest der FASTGOERES (aldrig glemmes). 0 = normalt.
        """
        self.eval()
        for _ in range(max_new_tokens):
            T = idx.size(1)
            if T <= self.config.block_size:
                idx_cond = idx
            elif context_len > 0:
                # Behold de fastgjorte kontekst-tegn + de nyeste tegn, saa modellen
                # aldrig "glemmer" konteksten -- heller ikke i lange historier.
                n_recent = self.config.block_size - context_len
                idx_cond = torch.cat([idx[:, :context_len], idx[:, -n_recent:]], dim=1)
            else:
                idx_cond = idx[:, -self.config.block_size:]   # klip fra venstre (glemmer det gamle)
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## 4. Sådan træner vi modellen

Træning er en løkke: hent et stykke tekst -> lad modellen gætte næste tegn -> mål hvor forkert
den var (**loss**) -> justér dens tal en lille smule. Det gentages mange gange.

Vi måler jævnligt loss på både `train`-data og `val`-data (tekst modellen ikke træner på). Hvis
val-loss ikke bliver bedre i et stykke tid, stopper vi (*early stopping*). Funktionen `train_model`
bruges **både** til basis-træning og finetuning -- det er samme slags træning.

In [7]:
def get_batch(data_tensor):
    """Traekker BATCH_SIZE tilfaeldige tekststykker af laengde BLOCK_SIZE."""
    ix = torch.randint(len(data_tensor) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data_tensor[i:i + BLOCK_SIZE]       for i in ix])
    y = torch.stack([data_tensor[i + 1:i + BLOCK_SIZE + 1] for i in ix])  # y = x forskudt ét tegn
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def estimate_loss(model, train_data, val_data, eval_iters=None):
    """Maaler gennemsnitlig loss paa train og val (uden at traene)."""
    if eval_iters is None:
        eval_iters = EVAL_ITERS
    out = {}
    model.eval()
    for name, d in [('train', train_data), ('val', val_data)]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(d)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[name] = losses.mean().item()
    model.train()
    return out

def train_model(model, train_data, val_data, max_iters, learning_rate, ckpt_path):
    """Traener modellen og gemmer den bedste version (lavest val-loss) til ckpt_path."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(BETA1, BETA2))
    best_val = float('inf')
    no_improve = 0
    history = {'iter': [], 'train': [], 'val': []}

    for it in tqdm(range(max_iters), desc="Traener"):
        # Maal og gem joevnligt.
        if it % EVAL_INTERVAL == 0 or it == max_iters - 1:
            losses = estimate_loss(model, train_data, val_data)
            history['iter'].append(it)
            history['train'].append(losses['train'])
            history['val'].append(losses['val'])
            tqdm.write(f"Skridt {it:5d}: train loss {losses['train']:.3f} | val loss {losses['val']:.3f}")
            if losses['val'] < best_val:
                best_val = losses['val']
                no_improve = 0
                torch.save(model.state_dict(), ckpt_path)   # gem bedste model
            else:
                no_improve += 1
            if no_improve >= EARLY_STOPPING_PATIENCE:
                tqdm.write("Stopper tidligt (val-loss blev ikke bedre).")
                break

        # Ét traenings-skridt.
        xb, yb = get_batch(train_data)
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))  # behold den bedste
    return history

def plot_history(history, title="Traenings-loss"):
    """Tegner train- og val-loss."""
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4))
    plt.plot(history['iter'], history['train'], label='train')
    plt.plot(history['iter'], history['val'], label='val')
    plt.xlabel('Iteration'); plt.ylabel('Loss'); plt.title(title)
    plt.legend(); plt.grid(True); plt.show()

## 5. Basis-modellen

Her bygger vi modellen og skaffer en **basis-model** trænet på det seriøse fantasy-data.

- `RETRAIN_BASE = False` (standard): vi **loader** den færdige `base_model.pth` (den har
  underviseren typisk lavet på forhånd). Hurtigt!
- `RETRAIN_BASE = True`: vi **træner** basis fra bunden. Det tager længere tid, men I behøver
  **ikke slette nogen filer** -- bare sæt flaget til `True`.

In [8]:
# Byg modellen ud fra kontrolpanelets indstillinger.
config = GPTConfig(vocab_size=vocab_size, block_size=BLOCK_SIZE,
                   n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD, dropout=DROPOUT)
model = GPT(config).to(DEVICE)

BASE_CKPT = 'base_model.pth'
if RETRAIN_BASE or not os.path.exists(BASE_CKPT):
    grund = "RETRAIN_BASE=True" if RETRAIN_BASE else f"ingen '{BASE_CKPT}' fundet"
    print(f"Traener basis-modellen fra bunden ({grund}) paa data_base.txt ...\n")
    base_train, base_val = load_dataset('data_base.txt')
    base_history = train_model(model, base_train, base_val, MAX_ITERS, LEARNING_RATE, BASE_CKPT)
    plot_history(base_history, title="Basis-traening")
else:
    print(f"Loader faerdig basis-model fra '{BASE_CKPT}' (RETRAIN_BASE=False).")
    model.load_state_dict(torch.load(BASE_CKPT, map_location=DEVICE))

Modellen har 10.78 millioner parametre.
Loader faerdig basis-model fra 'base_model.pth' (RETRAIN_BASE=False).


## 6. Lad modellen skrive -- og giv den evt. kontekst

To knapper styrer hvor "vild" modellen er:
- **temperature**: lav (fx 0.5) = forsigtig og forudsigelig, høj (fx 1.2) = vild og kreativ.
- **top_k**: modellen vælger kun mellem de `top_k` mest sandsynlige tegn (mindre = mere "sikkert").

**Vil du give modellen en kontekst?** Modellen kan kun se de sidste `BLOCK_SIZE` tegn ad gangen og
"glemmer" derfor starten i lange historier. Skriver du noget i variablen `context` nedenfor, bliver
det **fastgjort** forrest, så det aldrig glemmes. Lad `context` stå tom (`""`) hvis du ikke vil bruge det.

In [9]:
def generate_story(model, story_start, context="", max_new_tokens=None,
                   temperature=None, top_k=None):
    """Genererer en historie i 'Navn: replik'-format.

    story_start : teksten historien starter med (fx 'Sir Aldric:').
    context     : valgfri baggrund der FASTGOERES forrest, saa den ikke glemmes. "" = ingen kontekst.
    """
    if max_new_tokens is None: max_new_tokens = GEN_TOKENS
    if temperature is None:    temperature = TEMPERATURE
    if top_k is None:          top_k = TOP_K

    context     = filter_to_vocab(context)[0]       # fjern evt. ukendte tegn
    story_start = filter_to_vocab(story_start)[0]
    context_ids = encode(context)

    # Konteksten maa hoejst fylde halvdelen af vinduet, saa der er plads til selve historien.
    max_ctx = model.config.block_size // 2
    if len(context_ids) > max_ctx:
        context_ids = context_ids[:max_ctx]
        print("(Konteksten blev forkortet, saa der er plads til selve historien.)")

    if context_ids:
        prompt_ids = context_ids + encode("\n") + encode(story_start)
        context_len = len(context_ids) + 1          # +1 for linjeskiftet
    else:
        prompt_ids = encode(story_start)
        context_len = 0

    idx = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    out = model.generate(idx, max_new_tokens, temperature=temperature,
                         top_k=top_k, context_len=context_len)
    return decode(out[0].tolist())


# --- Skriv en historie med BASIS-modellen ---
story_start = "Sir Aldric:"
context = ""   # valgfrit, fx: "[Kontekst: Sir Aldric og Heksen Yrsa leder efter det forsvundne sværd.]"

print(generate_story(model, story_start, context=context))

Sir Aldric: *lytter ved døren ind til den tågede sump*
[Kontekst: En gammel profeti siger at den glemte bog skal bringen til den tågede sump før vagtskiftet.]
Droning Sigrid: Vi må nå frem til den tågede sump før solnedgang.
Troldmanden Mira: *skjuler det forsvundne sværd under sin kape*
Bueskytten Kasper: Hold øje med skyggerne i det høje tårn.
Tyven Freja: *lytter ved døren ind til krypten*
Droning Sigrid: Tag den gyldne nøgle og løb, jeg holder dem tilbage.
Tyven Freja: Kongeriget regner med os nu.
Tyven Freja: Frygt ikke - vi er rastløse nok til det her.
Tyven Freja: *trækker sit sværd og spejder mod


## 7. JERES OPGAVE: Finetuning

Nu tager vi basis-modellen og giver den lidt ekstra træning på *nyt* data, så den skifter stil.
Vi starter altid fra basis-modellen og træner med **lavere learning rate** og **færre skridt** --
derfor går det hurtigt (under ~10 min på en T4).

- `DO_FINETUNE = True` (standard): finetun på det valgte data og gem `finetuned_model.pth`.
- `DO_FINETUNE = False`: spring finetuning over (genbrug en tidligere finetunet model hvis den findes).
  Så slipper I for at vente, hvis I bare vil generere igen.

I cellen vælger I `FINETUNE_FILE`. Standard er **data 2** (`data_finetune.txt`). Vil I have det
**helt vildt**, så skift til **data 3** (`data_cooked.txt`).

In [10]:
# Hvilket data vil du finetune paa?
FINETUNE_FILE = 'data_finetune.txt'   # data 2 (jokes). Skift til 'data_cooked.txt' for ren brainrot (data 3)!

# (Valgfrit) Tilfoej dit EGET data i samme format 'Navn: replik'. Lad staa tom "" hvis ikke.
EXTRA_TRAINING_TEXT = ""

FT_CKPT = 'finetuned_model.pth'
if DO_FINETUNE:
    print(f"Finetuner paa {FINETUNE_FILE} ...\n")
    model.load_state_dict(torch.load(BASE_CKPT, map_location=DEVICE))   # start ALTID fra basis
    ft_train, ft_val = load_dataset(FINETUNE_FILE, EXTRA_TRAINING_TEXT)
    ft_history = train_model(model, ft_train, ft_val, FINETUNE_ITERS, FINETUNE_LR, FT_CKPT)
    plot_history(ft_history, title="Finetuning-loss")
elif os.path.exists(FT_CKPT):
    print("DO_FINETUNE=False: loader tidligere finetunet model.")
    model.load_state_dict(torch.load(FT_CKPT, map_location=DEVICE))
else:
    print("DO_FINETUNE=False og ingen finetunet model fundet -- bruger basis-modellen.")

Finetuner paa data_finetune.txt ...



Traener:   0%|          | 0/1000 [00:07<?, ?it/s]

Skridt     0: train loss 4.560 | val loss 4.707


Traener:   0%|          | 0/1000 [00:07<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 3.65 GiB of which 6.06 MiB is free. Including non-PyTorch memory, this process has 3.63 GiB memory in use. Of the allocated memory 3.52 GiB is allocated by PyTorch, and 26.88 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Generér nu med den finetunede model og sammenlign med basis-modellen ovenfor. God fornøjelse!

In [ ]:
# --- Skriv en historie med den FINETUNEDE model ---
story_start = "Sir Aldric:"
context = ""   # proev fx en kontekst her

print(generate_story(model, story_start, context=context))